# 1. Baseline : NLLB-200 (zero-shot) FR ↔ Éwé

**Objectif** : mesurer la qualité de traduction du modèle **NLLB-200-distilled-600M**
(Meta AI) sur notre corpus de test **sans aucun entraînement** (mode "zero-shot").

C'est la **référence de départ** : tout le travail de fine-tuning (notebook 2)
devra faire mieux que ces scores.

## Comment ça marche ?

- **NLLB** ("No Language Left Behind") est un modèle de traduction multilingue
  entraîné sur 200 langues, dont l'**éwé** (code `ewe_Latn`).
- Il est **"zero-shot"** pour nous : il n'a jamais vu notre corpus, mais il a vu
  de l'éwé pendant son entraînement.
- On mesure la qualité avec deux métriques standard :
  - **chrF++** (la métrique principale du projet, robuste aux petites variations)
  - **BLEU** (métrique classique, plus stricte)

> ⚠️ Le test set est chargé depuis le **repo GitHub public** du projet.
> C'est le split `test.tsv` : 1 898 paires jamais utilisées pour l'entraînement.

In [ ]:
# Installation des bibliothèques nécessaires
# - transformers : modèles HuggingFace (NLLB)
# - sacrebleu    : métriques chrF++ et BLEU
# - pandas       : lecture des fichiers TSV
# - sentencepiece : tokenizer de NLLB (obligatoire)
!pip install -q transformers sacrebleu pandas sentencepiece datasets

print("✅ Dépendances installées")

In [ ]:
# Imports + détection du GPU
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Colab met un GPU (T4) à disposition : on l'utilise si disponible.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔧 Device utilisé :", device)
print("   (cuda = GPU, cpu = lent mais fonctionne)")

In [ ]:
# Chargement du jeu de test depuis le repo GitHub public
# Les données du projet sont versionnées : ce notebook charge la version "main".
URL_TEST = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/data/processed/v0.1/test.tsv"

try:
    df = pd.read_csv(URL_TEST, sep="\t")
    print(f"✅ Test set chargé : {len(df)} paires FR↔Éwé")
    print(df.head(3))
except Exception as e:
    print("❌ Téléchargement GitHub impossible :", e)
    print("→ Solution : télécharge test.tsv depuis le repo et exécute cette cellule :")
    print("   from google.colab import files; upload = files.upload()")

In [ ]:
# Chargement du modèle NLLB-200-distilled-600M
# 600M paramètres = version "distilled" (légère), parfaite pour un GPU gratuit.
MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Vérification : les codes de langue existent-ils ?
assert "fra_Latn" in tokenizer.additional_special_tokens, "français absent ?!"
assert "ewe_Latn" in tokenizer.additional_special_tokens, "éwé absent ?!"
print("✅ Modèle chargé — codes langue : fra_Latn (fr), ewe_Latn (éwé)")

In [ ]:
# Fonction de traduction en batch
# - src / tgt : codes de langue NLLB (fra_Latn, ewe_Latn)
# - num_beams=4 : recherche en faisceau (meilleure qualité que greedy)
# - Le tokenizer doit connaître la langue SOURCE avant d'encoder.

def traduire(textes, src="fra_Latn", tgt="ewe_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = src          # langue source pour l'encodage
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),  # langue cible
                max_new_tokens=max_len,
                num_beams=4,
            )
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

print("✅ Fonction de traduction prête")

In [ ]:
# Évaluation FR → ÉWÉ (le sens qui nous intéresse le plus)
# On traduit les 1 898 phrases françaises du test set, puis on compare
# aux traductions éwé de référence avec chrF++ et BLEU.

preds_fr_ee = traduire(df["fr"].tolist(), src="fra_Latn", tgt="ewe_Latn")
refs_ee = df["ewe"].tolist()

chrf_fr_ee = sacrebleu.corpus.chrf(preds_fr_ee, [refs_ee])
bleu_fr_ee = sacrebleu.corpus.bleu(preds_fr_ee, [refs_ee])

print("📊 FR → ÉWÉ (zero-shot)")
print(f"   chrF++ : {chrf_fr_ee.score:.2f}")
print(f"   BLEU   : {bleu_fr_ee.score:.2f}")

# Afficher 3 exemples concrets pour voir la qualité à l'œil
for i in range(3):
    print(f"\n--- Exemple {i+1} ---")
    print(f"FR : {df['fr'].iloc[i]}")
    print(f"Réf: {refs_ee[i]}")
    print(f"Préd: {preds_fr_ee[i]}")

In [ ]:
# Évaluation ÉWÉ → FR (sens inverse)
# Utile pour vérifier que le modèle comprend aussi l'éwé en entrée.

preds_ee_fr = traduire(df["ewe"].tolist(), src="ewe_Latn", tgt="fra_Latn")
refs_fr = df["fr"].tolist()

chrf_ee_fr = sacrebleu.corpus.chrf(preds_ee_fr, [refs_fr])
bleu_ee_fr = sacrebleu.corpus.bleu(preds_ee_fr, [refs_fr])

print("📊 ÉWÉ → FR (zero-shot)")
print(f"   chrF++ : {chrf_ee_fr.score:.2f}")
print(f"   BLEU   : {bleu_ee_fr.score:.2f}")

print("\n📋 Tableau de bord baseline :")
print(f"   FR→ÉWÉ : chrF++ {chrf_fr_ee.score:.2f} | BLEU {bleu_fr_ee.score:.2f}")
print(f"   ÉWÉ→FR : chrF++ {chrf_ee_fr.score:.2f} | BLEU {bleu_ee_fr.score:.2f}")

## Comment interpréter ces scores ?

- **chrF++ ~40-55** sur cette tâche = le modèle "se débrouille" (vocabulaire
  religieux bien connu de NLLB, car la bible fait partie de ses données).
- **BLEU bas (< 15)** est normal : BLEU est très strict sur les mots exacts,
  et l'éwé de 1913 a une orthographe différente de l'éwé moderne vu par NLLB.
- Ces scores sont notre **référence** : le notebook 2 (fine-tuning LoRA sur
  notre corpus) doit les **dépasser**, surtout en chrF++.

> 💡 Si le score est très bas, vérifie que le GPU est actif
> (menu *Exécution > Changer le type d'exécution > T4 GPU*).